# A Psychological Object Model Of The ABA "Frequent Fliers"

- TODO: expand bullets

- American Bar: "Parents preoccupied with identity reconstruction"
- Narcissistic Personality Disorder: "If they are male (and most are)"
- "Psychology’s Replication Crisis Is Running Out of Excuses"
- Soviet Stalin: "Show me the man and I’ll find you the crime"

Once again, we have already too much accumulated code to load into our notebook. For your refence, we sample from the attached local source files: *datasets.py, layers.py, main.py, modules.py, samples.py, tasks.py, trafo.py* and *utils.py*.

- our previous blog has already generated the necessary `small` set of  samples
- if you have not run the code yet, please follow that blog to generate samples

- in this blog we focus on `yes-no-extended` (YNX), `masked-extended` (MSX) and `faulty` (FIX) samples
- describe samples

- we have mentioned already that we are largely reusing the model from the first section of our blogs
- in this blog we provide more details of the changes in the layers
- we start with a significant change in our `Layer` base class, we have started to support the Keras `get_config` method to be able to recreate our layers from checkpoints
- in `utils.py` we extend our `Params` class with a new `Config` class that encapsulates the actually persisted param values

In [1]:
class Params:
    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)

    def cfg_items(self, *keys):
        for k in keys:
            yield k, getattr(self, k, None)


class Config(Params):
    runtime = Params(is_training=True, print_toks=False)

    def __init__(self, **kw):
        super().__init__(**kw)
        self._items = kw.items()

    def items(self):
        return self._items

- using the new `Config` we then add support for the "config" mechanism in `Layer`:

In [3]:
import tensorflow as tf
ks = tf.keras
ki = ks.initializers

class Layer(ks.layers.Layer):
    initer = None

    @staticmethod
    def cfg_items(ps):
        yield from ps.cfg_items('initer_stddev', )

    @classmethod
    def from_config(cls, cfg):
        return cls(qu.Config(**cfg))

    def __init__(self, ps, **kw):
        kw.setdefault('name', qu.to_snake_case(type(self).__name__))
        kw.setdefault('dtype', tf.float32)
        super().__init__(**kw)
        if isinstance(ps, qu.Config):
            self.cfg = ps
        else:
            self.cfg = qu.Config(**dict(self.cfg_items(ps)))
        cfg = self.cfg
        if cfg.initer_stddev:
            self.initer = ki.TruncatedNormal(stddev=cfg.initer_stddev)

    def get_config(self):
        b = super().get_config().items()
        c = self.cfg.items()
        return dict(list(b) + list(c))

- to demonstrate how just the necessary params are saved for each layer, we provide here a fragment of the already presented `Frames` class
- the rest of the class as well as the derived `Tokens` and `Meta` classes are mostly unchanged
- the significant difference is that we have started to use the attached instance of the `Config` class instead of the previously used bare `Params` 

In [4]:
class Frames(Layer):
    def __init__(self, ps):
        super().__init__(ps, dtype=tf.int32)
        s = [self.cfg.dim_batch, self.cfg.width_enc]
        kw = dict(initializer='zeros', trainable=False, use_resource=True)
        self.prev = self.add_weight('prev', s, **kw)

    def cfg_items(self, ps):
        yield from super().cfg_items(ps)
        yield from ps.cfg_items(
            'dim_batch',
            'dim_hist',
            'width_dec',
            'width_enc',
        )

- another addition to the layers in this blog is the addition of `input_signatures` to the `call` methods' `tf.function` definitions
- since we keep a tight control of the shapes of our "transfer" tensors, there is no need for TF to unnecessarily generate Python code
- we provide the partial `Decode` class example to illustrate this addition

In [5]:
class Decode(Layer):
    @tf.function(input_signature=[[
        tf.TensorSpec(shape=[None, None, None]),
        tf.TensorSpec(shape=[None], dtype=tf.int32),
        tf.TensorSpec(shape=[None, None, None])
    ]])
    def call(self, x):
        y, ye = x[:-1], x[-1]
        for dec in self.decs:
            y = dec(y + [ye])
        return y

- we also added a new layer to specifically help with the MSK and MSX samples
- the `Deduce` layer supplies a graph op to conditionally loop through all the masked, by the `?` token, positions and one-by-one deduce the target characters
- the loops obviously are driven by the actual number of masked positions in the individual batches of samples
- a partial definition of the class is as follows:

In [6]:
class Deduce(Layer):
    def __init__(self, ps, embed, decode):
        super().__init__(ps)
        self.embed = embed
        self.decode = decode
        s = [self.cfg.dim_hidden, self.cfg.dim_vocab]
        self.inflate = qm.Dense(self, 'inflate', s)

    @tf.function
    def call(self, x):
        toks, *x = x
        if self.cfg.runtime.print_toks:
            qu.print_toks(toks, qd.vocab)
        y = self.deduce([toks] + x)
        n = tf.shape(y)[1]
        p = tf.shape(toks)[1] - n
        for i in tf.range(n):
            t = toks[:, :n]
            m = tf.equal(t, qd.MSK)
            if tf.equal(tf.reduce_any(m), True):
                t = self.update(t, m, y)
                if self.cfg.runtime.print_toks:
                    qu.print_toks(t, qd.vocab)
                toks = tf.pad(t, [[0, 0], [0, p]])
                y = self.deduce([toks] + x)
            else:
                e = tf.equal(t, qd.EOS)
                e = tf.math.count_nonzero(e, axis=1)
                if tf.equal(tf.reduce_any(tf.not_equal(e, 1)), False):
                    break
        return y

    def deduce(self, x):
        y = self.embed(x[:-1])
        y, lens = self.decode(y + x[-1:])
        y = self.inflate(y)
        y = tf.RaggedTensor.from_tensor(y, lens).to_tensor()
        return y

    def update(self, toks, msks, ctx):
        i = tf.cast(msks, tf.int32)
        i = tf.argmax(i, axis=1, output_type=tf.int32)
        n = tf.shape(msks)[0]
        i = tf.stack([tf.range(n), i], axis=1)
        m = tf.zeros_like(msks)
        m = tf.tensor_scatter_nd_update(m, i, tf.ones([n], tf.bool))
        y = tf.boolean_mask(ctx, m)
        y = tf.math.log_softmax(y)
        y = tf.argmax(y, axis=-1, output_type=tf.int32)
        y = tf.tensor_scatter_nd_update(toks, i, y)
        y = tf.where(tf.logical_and(msks, m), y, toks)
        return y

- with these changes in the code we can adjust the groups of samples we want to train on and run the previously presented steps
- in this blog we skip re-running the training sessions
- this concludes our blog, please click on the next blog for further details of the additional changes